# Gradient Descent

This notebook accompanies the **ML Viz** lesson on Gradient Descent.
We build SGD, Momentum, RMSprop, and Adam from scratch and visualize their trajectories.

**Companion lesson:** https://ml-viz.vercel.app/courses/neural-networks/02-gradient-descent

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## The loss surface

We use a simple 2D quadratic loss:

$$L(w_1, w_2) = 0.4 w_1^2 + 0.9 w_2^2 + 0.1 w_1 w_2$$

The global minimum is at $(0, 0)$.  The asymmetric curvature (different eigenvalues) makes it a good test for optimizers.

In [ ]:
def loss(w):
    """2D quadratic bowl with asymmetric curvature."""
    w1, w2 = w
    return 0.4 * w1**2 + 0.9 * w2**2 + 0.1 * w1 * w2

def grad(w):
    """Analytical gradient of loss."""
    w1, w2 = w
    return np.array([0.8 * w1 + 0.1 * w2,
                     1.8 * w2 + 0.1 * w1])

# Plot the surface
w1s = np.linspace(-4, 4, 200)
w2s = np.linspace(-4, 4, 200)
W1, W2 = np.meshgrid(w1s, w2s)
Z = 0.4 * W1**2 + 0.9 * W2**2 + 0.1 * W1 * W2

fig, ax = plt.subplots(figsize=(7, 6))
cs = ax.contourf(W1, W2, Z, levels=25, cmap='YlOrRd', alpha=0.75)
plt.colorbar(cs, ax=ax, label='Loss')
ax.set_xlabel('w₁'); ax.set_ylabel('w₂')
ax.set_title('Loss Surface L(w₁, w₂)', color='white')
plt.tight_layout(); plt.show()

## Implementing optimizers from scratch

Each optimizer is a function `step(w, g, state) → (w_new, state_new)`.

In [ ]:
def sgd_step(w, g, state, lr=0.1):
    """Vanilla SGD: w ← w - lr * g"""
    return w - lr * g, state


def momentum_step(w, g, state, lr=0.1, beta=0.9):
    """SGD with Momentum: accumulates velocity."""
    v = state.get('v', np.zeros_like(w))
    v = beta * v + (1 - beta) * g
    return w - lr * v, {'v': v}


def rmsprop_step(w, g, state, lr=0.05, beta=0.9, eps=1e-8):
    """RMSprop: adaptive learning rates via squared gradient EMA."""
    s = state.get('s', np.zeros_like(w))
    s = beta * s + (1 - beta) * g**2
    return w - lr * g / (np.sqrt(s) + eps), {'s': s}


def adam_step(w, g, state, lr=0.1, beta1=0.9, beta2=0.999, eps=1e-8):
    """Adam: combines momentum and RMSprop with bias correction."""
    t = state.get('t', 0) + 1
    m = state.get('m', np.zeros_like(w))
    v = state.get('v', np.zeros_like(w))
    m = beta1 * m + (1 - beta1) * g
    v = beta2 * v + (1 - beta2) * g**2
    m_hat = m / (1 - beta1**t)   # bias correction
    v_hat = v / (1 - beta2**t)
    return w - lr * m_hat / (np.sqrt(v_hat) + eps), {'t': t, 'm': m, 'v': v}


def run_optimizer(step_fn, w0, n_steps=60, **kwargs):
    w, state = np.array(w0, dtype=float), {}
    path = [w.copy()]
    losses = [loss(w)]
    for _ in range(n_steps):
        g = grad(w)
        w, state = step_fn(w, g, state, **kwargs)
        path.append(w.copy())
        losses.append(loss(w))
    return np.array(path), losses

In [ ]:
w0 = [3.5, 3.0]
results = {
    'SGD':      run_optimizer(sgd_step,      w0, lr=0.15),
    'Momentum': run_optimizer(momentum_step, w0, lr=0.1),
    'RMSprop':  run_optimizer(rmsprop_step,  w0, lr=0.12),
    'Adam':     run_optimizer(adam_step,     w0, lr=0.3),
}
colors = {'SGD': '#f97316', 'Momentum': '#818cf8', 'RMSprop': '#eab308', 'Adam': '#14b8a6'}

fig, (ax_path, ax_loss) = plt.subplots(1, 2, figsize=(14, 5.5))

# Trajectory plot
ax_path.contourf(W1, W2, Z, levels=25, cmap='Greys', alpha=0.5)
for name, (path, _) in results.items():
    ax_path.plot(path[:, 0], path[:, 1], '-o', markersize=3,
                 color=colors[name], label=name, linewidth=1.8)
ax_path.scatter(0, 0, s=120, color='white', zorder=10, label='Minimum')
ax_path.set_title('Optimization Trajectories', color='white')
ax_path.legend()
ax_path.set_xlim(-4, 4); ax_path.set_ylim(-4, 4)

# Loss curves
for name, (_, losses) in results.items():
    ax_loss.semilogy(losses, color=colors[name], label=name, linewidth=2)
ax_loss.set_xlabel('Step'); ax_loss.set_ylabel('Loss (log scale)')
ax_loss.set_title('Convergence', color='white')
ax_loss.legend()

plt.tight_layout(); plt.show()

## Effect of learning rate

Too small = slow convergence. Too large = divergence.

In [ ]:
lrs = [0.01, 0.1, 0.5, 0.9]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, lr in zip(axes, lrs):
    path, losses = run_optimizer(sgd_step, w0, n_steps=80, lr=lr)
    ax.contourf(W1, W2, Z, levels=20, cmap='Greys', alpha=0.5)
    ax.plot(path[:, 0], path[:, 1], '-o', markersize=2.5, color='#818cf8', linewidth=1.5)
    ax.scatter(0, 0, s=80, color='#14b8a6', zorder=10)
    final = losses[-1]
    ax.set_title(f'lr={lr}  →  L={final:.4f}', color='white', fontsize=10)
    ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)

plt.suptitle('SGD with Different Learning Rates', color='white', y=1.02)
plt.tight_layout(); plt.show()

## Momentum and the optimizer family

Plain SGD can crawl through ravines. **Momentum** accumulates a velocity that smooths the path; **Adam** adds per-parameter adaptive step sizes.

In [ ]:
def gd(grad, x0, lr=0.1, steps=50, momentum=0.0):
    x, v, path = x0, 0.0, [x0]
    for _ in range(steps):
        v = momentum * v - lr * grad(x)
        x = x + v
        path.append(x)
    return np.array(path)

# Minimize f(x) = x^2  (grad = 2x)
grad = lambda x: 2 * x
for m in [0.0, 0.9]:
    p = gd(grad, 5.0, lr=0.1, momentum=m)
    print(f'momentum={m}: reached {p[-1]:.4f} in {len(p)-1} steps')

## Key takeaways

- Gradient descent steps **downhill**: $w \leftarrow w - \eta\,\nabla L$.
- The **learning rate** $\eta$ trades speed for stability — too big diverges, too small crawls.
- **Momentum** accelerates along consistent directions and damps oscillation.
- **Adam** (adaptive + momentum) is the safe default for deep learning.